# ALOE Shard L2 — LiquidAI/LFM3-1.7B (b0 Search & 3-Seed Confirm | NVIDIA L4)

**Mobile 1-Tap Instructions (~30 seconds):**
1. Ensure `HF_TOKEN` is saved once in Colab **Secrets** (key icon on the left sidebar, *Notebook access* toggled ON).
2. Tap the **Play** button on the single cell below and tap **Grant access**.
3. Once output shows `[SHARD LIVE] ... Starting trials...` (~25s), **close your browser / lock your phone**.
- **Pinned Code SHA:** `8187255cae07` | **Arms:** `b0` | **Phases:** `search,confirm` | **ETA:** `~6.5-8h on 1x NVIDIA L4`
- Auto-uploads every completed trial to private `Walapalam/aloe-ipd-v1-runs` and **auto-disconnects the L4 GPU** from the notebook kernel when finished.


In [ ]:
# @title Run L2_lfm3_b0 (~6.5-8h on 1x NVIDIA L4)
FORCE_STEAL_LEASE = False  # @param {type:"boolean"}
import getpass, os, subprocess, sys, time, torch

def _unassign_colab_kernel(delay_sec: float) -> None:
    try:
        from google.colab import runtime
        print(f"[COLAB] Disconnecting L4 runtime in {delay_sec:.0f}s to conserve compute units...", flush=True)
        time.sleep(delay_sec)
        runtime.unassign()
    except Exception:
        pass

# 1. Fail-closed GPU check (<1s before installing or burning compute units)
assert torch.cuda.is_available(), (
    "No GPU attached! Tap Runtime -> Change runtime type -> L4 GPU -> Save, then tap Play again."
)
gpu_name = torch.cuda.get_device_name(0)
assert gpu_name == "NVIDIA L4", (
    f"ADR-019 requires exact GPU 'NVIDIA L4', got '{gpu_name}'. "
    "Tap Runtime -> Change runtime type -> L4 GPU -> Save, then tap Play again."
)
print(f"[1/4] GPU verified: {gpu_name} | PyTorch {torch.__version__}", flush=True)

# 2. Read HF_TOKEN from Colab Secrets (or getpass fallback)
hf_token = ""
try:
    from google.colab import userdata
    hf_token = (userdata.get("HF_TOKEN") or "").strip()
except Exception:
    pass
if not hf_token:
    hf_token = getpass.getpass("Paste Hugging Face write token (hf_...): ").strip()
assert hf_token.startswith("hf_"), "Valid HF_TOKEN starting with hf_ is required."
os.environ["HF_TOKEN"] = hf_token
os.environ["HUGGING_FACE_HUB_TOKEN"] = hf_token
print("[2/4] HF_TOKEN loaded securely.", flush=True)

rc = 1
try:
    # 3. Install frozen user-space dependencies (preserving Colab pre-installed torch+cu130)
    subprocess.run([
        sys.executable, "-m", "pip", "install", "-q",
        "transformers==5.24.0", "peft==0.22.0", "botorch==0.18.1",
        "accelerate>=1.0.0", "datasets>=3.0.0", "huggingface_hub>=0.26.0",
        "pydantic>=2.9.0", "rich>=13.9.0", "typer>=0.12.5"
    ], check=True)
    print("[3/4] Dependencies ready. Preparing isolated shard worktree...", flush=True)

    REPO_URL = "https://github.com/Walapalam/aloe.git"
    ORCH_DIR = "/content/aloe_orch_L2_lfm3_b0"
    WORK_DIR = "/content/aloe_work_L2_lfm3_b0"
    PINNED_SHA = "8187255cae076f583dce455acaf5019b20bad1e4"

    if not os.path.exists(ORCH_DIR):
        subprocess.run(["git", "clone", "-q", "-b", "feat/ipd-protocol-v1", REPO_URL, ORCH_DIR], check=True)
    else:
        subprocess.run(["git", "-C", ORCH_DIR, "fetch", "-q", "origin", "feat/ipd-protocol-v1"], check=True)
        subprocess.run(["git", "-C", ORCH_DIR, "reset", "--hard", "origin/feat/ipd-protocol-v1"], check=True)

    if not os.path.exists(WORK_DIR):
        subprocess.run(["git", "clone", "-q", "-b", "feat/ipd-protocol-v1", REPO_URL, WORK_DIR], check=True)
    else:
        subprocess.run(["git", "-C", WORK_DIR, "fetch", "-q", "origin", "feat/ipd-protocol-v1"], check=True)
    # Reset tracked files ONLY (never git clean -fdq so mid-trial checkpoints survive local re-runs)
    subprocess.run(["git", "-C", WORK_DIR, "checkout", "-q", "--detach", PINNED_SHA], check=True)
    subprocess.run(["git", "-C", WORK_DIR, "reset", "--hard", PINNED_SHA], check=True)

    print("[4/4] Launching atomic shard runner (safe to close phone once [SHARD LIVE] prints)...", flush=True)
    cmd = [
        sys.executable, "-u", f"{ORCH_DIR}/scripts/run_colab_shard.py",
        "--shard-id", "L2_lfm3_b0",
        "--model", "LiquidAI/LFM3-1.7B",
        "--work-repo", WORK_DIR,
        "--expected-git-sha", PINNED_SHA,
        "--arms", "b0",
        "--phases", "search,confirm",
        "--search-seed", "42",
        "--pinned-gpu", "L4",
        "--expected-exact-gpu", "NVIDIA L4",
        "--hf-repo", "Walapalam/aloe-ipd-v1-runs",
    ]
    if FORCE_STEAL_LEASE:
        cmd.append("--force-steal-lease")
    proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    assert proc.stdout is not None
    for line in proc.stdout:
        print(line, end="", flush=True)
    rc = proc.wait()
    if rc != 0:
        raise RuntimeError(f"Shard runner exited with status {rc}")
finally:
    _unassign_colab_kernel(3.0 if rc == 0 else 25.0)
